# Notebook 05 — SHAP Attribution
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/05_shap_interpretation.ipynb)

GradientSHAP attribution of LSTM predictions and catchment-level surrogate model explaining which physical characteristics drive DO predictability. Requires notebooks 03 and 04.

# AareML — 05 SHAP Interpretation

**Project:** AareML — Predicting River Water Quality in Swiss Catchments

---

## Overview

SHAP (SHapley Additive exPlanations) answers two questions:

1. **Input-feature importance:** Which sensor inputs and which lookback
   days drive the LSTM's DO forecast? (DeepSHAP / GradientSHAP on the model)

2. **Catchment-attribute importance:** Across all gauges, which catchment
   characteristics (land cover, area, elevation) predict prediction skill?
   (TreeSHAP on a gradient-boosted surrogate trained on `attribute → RMSE`)

Together these answer the interpretability question at the heart of AareML:
*what makes a catchment's DO predictable by an LSTM?*


## 0 · Setup


In [1]:
# ── Colab setup (auto-runs only in Google Colab) ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    from pathlib import Path

    # ── 1. Clone repo ──────────────────────────────────────────────────────
    if not Path('AareML').exists():
        os.system('git clone https://github.com/polar-bear-after-lunch/AareML.git')
    if not str(Path.cwd()).endswith('AareML'):
        os.chdir('AareML')

    # ── 2. Install dependencies ────────────────────────────────────────────
    os.system('pip install -q -r requirements.txt')

    # ── 3. Mount Google Drive for persistent data storage ─────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    # Data stored in Drive so it survives across sessions (~360 MB total)
    DRIVE_DATA = Path('/content/drive/MyDrive/AareML_data')
    LOCAL_DATA = Path('data')
    LOCAL_DATA.mkdir(exist_ok=True)

    # ── 4. CAMELS-CH-Chem ─────────────────────────────────────────────────
    DRIVE_CAMELS = DRIVE_DATA / 'camels-ch-chem'
    LOCAL_CAMELS = LOCAL_DATA / 'camels-ch-chem'

    if DRIVE_CAMELS.exists():
        # Already in Drive — symlink to local path (fast, no re-download)
        if not LOCAL_CAMELS.exists():
            os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem loaded from Google Drive.')
    else:
        # First time — download to Drive
        print('Downloading CAMELS-CH-Chem to Google Drive (~165 MB, one-time)...')
        DRIVE_DATA.mkdir(parents=True, exist_ok=True)
        os.system(
            'wget -q --show-progress -O /tmp/camels.zip '
            '"https://zenodo.org/api/records/14980027/files/camels-ch-chem.zip/content"'
        )
        os.system(f'unzip -q /tmp/camels.zip -d {DRIVE_CAMELS}')
        os.system('rm /tmp/camels.zip')
        os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem saved to Google Drive for future sessions.')

    print(f'Setup complete. Working directory: {os.getcwd()}')


In [2]:
# ── CPU thread optimisation ────────────────────────────────────────────────
# PyTorch LSTM on CPU is fastest with 4-6 threads — beyond that,
# thread synchronisation overhead outweighs the parallelism gains.
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)  # clamp to 6 — empirically optimal on macOS
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} logical | Using {N_THREADS} threads for PyTorch')


CPU cores: 128 logical | Using 6 threads for PyTorch


In [3]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader

from src.config  import *
from src.data    import load_gauge, load_metadata, preprocess, train_val_test_split, make_windows
from src.model   import (
    RiverDataset, Seq2SeqLSTM,
    get_y_true, load_checkpoint,
)
from src.metrics import mean_rmse

from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


In [4]:
# ── GPU / DataLoader configuration ────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# LOCAL_TEST: True = fast CPU verification, False = full run
LOCAL_TEST = DEVICE.type == 'cpu'
if LOCAL_TEST:
    print('LOCAL_TEST mode: reduced epochs/trials for quick verification')


Device: cuda
GPU: NVIDIA GeForce RTX 4090
VRAM: 23.8 GB


## 1 · Load Model and Data

Load the best single-site LSTM from notebook 03.


In [5]:
# ── Constants ─────────────────────────────────────────────────────────────
N_FEAT = len(FEATURES)   # number of input features (4)
N_TGT  = len(TARGETS)    # number of output targets (2)
print(f'N_FEAT={N_FEAT}, N_TGT={N_TGT}')


N_FEAT=4, N_TGT=2

In [6]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)
train, val, test = train_val_test_split(data)
# Compute training means without duplicated index (FEATURES and TARGETS overlap)
train_means = (
    pd.concat([train[FEATURES].mean(), train[TARGETS].mean()])
    .groupby(level=0).first()   # deduplicate temp_sensor / O2C_sensor
)

# Scalers — refit with same logic as notebook 03
def impute(df, means): return df.fillna(means)

# Load checkpoint FIRST, then reconstruct scalers from it
ckpt_path = RESULTS_DIR / 'lstm_single_site_best.pt'
if ckpt_path.exists():
    ckpt = load_checkpoint(ckpt_path, device=DEVICE)
    print(f'Loaded checkpoint. Params: {ckpt["best_params"]}')
else:
    raise FileNotFoundError(
        f'Checkpoint not found: {ckpt_path}\n'
        'SHAP attributions on an untrained model are meaningless.\n'
        'Please run notebook 03 (job_03_lstm.sh) first.'
    )

# Reconstruct scalers from checkpoint (matches training exactly)
from src.model import reconstruct_scalers
if ckpt is not None:
    feat_scaler, tgt_scaler = reconstruct_scalers(ckpt)
    print(f'Scalers reconstructed from checkpoint.')
    print(f'feat_scaler mean: {feat_scaler.mean_}')
    print(f'tgt_scaler  mean: {tgt_scaler.mean_}')
else:
    from sklearn.preprocessing import StandardScaler
    feat_scaler = StandardScaler().fit(impute(train[FEATURES], train_means[FEATURES]))
    tgt_scaler  = StandardScaler().fit(impute(train[TARGETS],  train_means[TARGETS]))
    print('Using freshly fitted scalers (checkpoint not available).')

def scale_windows_fn(X_raw, y_raw):
    N, L, F = X_raw.shape; Nt, H, T = y_raw.shape
    X_s = feat_scaler.transform(X_raw.reshape(-1,F)).reshape(N,L,F).astype(np.float32)
    y_s = tgt_scaler.transform(y_raw.reshape(-1,T)).reshape(Nt,H,T).astype(np.float32)
    return X_s, y_s

X_test, y_test, d_test = make_windows(test, train_means)
Xs_test, ys_test = scale_windows_fn(X_test, y_test)
ds_test  = RiverDataset(Xs_test, ys_test)

# Build model from checkpoint
if ckpt is not None:
    bp   = ckpt['best_params']
    model = Seq2SeqLSTM(N_FEAT, N_TGT, hidden=bp['hidden'],
                        n_layers=bp['n_layers'], dropout=bp['dropout'])
    model.load_state_dict(ckpt['model_state'])
else:
    model = Seq2SeqLSTM(N_FEAT, N_TGT)

model = model.eval().to(DEVICE)
print(f'Test windows: {len(ds_test):,}')


[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461


[model] load_checkpoint: loaded from /storage/homefs/tn20y076/AareML/notebooks/../results/lstm_single_site_best.pt, keys=['model_state', 'best_params', 'feat_scaler_mean', 'feat_scaler_scale', 'tgt_scaler_mean', 'tgt_scaler_scale']
Loaded checkpoint. Params: {'hidden': 256, 'n_layers': 2, 'dropout': 0.08003951267737228, 'lr': 0.001004753025365132, 'batch_size': 128, 'teacher_forcing_start': 0.6487283159585305}
Scalers reconstructed from checkpoint.
feat_scaler mean: [  7.97703414   8.10441397 294.74799341  11.31732532]
tgt_scaler  mean: [11.31732532  7.97703414]
[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18


[model] RiverDataset: 1348 samples, X=(1348, 21, 4), y=(1348, 14, 2)


Test windows: 1,348


## 2 · GradientSHAP on LSTM Inputs

GradientSHAP (a gradient × input method from `captum`) attributes
each of the 21 × 4 = 84 input features to the LSTM's **1-day-ahead DO
forecast**. We use 200 random background samples as the baseline.

> *Note:* `captum` must be installed: `pip install captum`


In [7]:
# Ensure training mode for cuDNN RNN gradient computation
model.train()

try:
    import captum
    from captum.attr import GradientShap
    HAS_CAPTUM = True
    print(f'captum {captum.__version__} available')
except ImportError:
    HAS_CAPTUM = False
    print('captum not installed — run: pip install captum')
    print('Falling back to gradient × input approximation.')


captum 0.7.0 available


In [8]:
# ── Wrapper: model predicts step 0 (1-day-ahead) of DO target only ────────
class DO1DayWrapper(torch.nn.Module):
    """Wraps the seq2seq model, returning only the 1-day-ahead DO scalar."""
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

    def forward(self, x):
        out = self.base(x)          # [batch, horizon, n_tgt]
        return out[:, 0, 0]         # 1-day-ahead DO


wrapper = DO1DayWrapper(model).eval().to(DEVICE)

# Background: 200 random training windows
X_train_np, _, _ = make_windows(train, train_means)
X_train_s, _     = scale_windows_fn(X_train_np, np.zeros((len(X_train_np), HORIZON, N_TGT)))
bg_idx  = np.random.default_rng(SEED).choice(len(X_train_s), size=200, replace=False)
bg_tensor = torch.tensor(X_train_s[bg_idx]).to(DEVICE)

# Inputs: first 500 test windows
# Scale SHAP windows to available hardware — more = more stable attributions
n_explain_max = 500 if DEVICE.type == 'cuda' else (10 if LOCAL_TEST else 500)  # 500 uses ~3.7 GiB on RTX 4090 (safe below 24 GB limit)
n_explain = min(n_explain_max, len(Xs_test))
print(f'SHAP windows: {n_explain} ({"GPU" if DEVICE.type == "cuda" else "CPU"})')
inp_tensor = torch.tensor(Xs_test[:n_explain]).to(DEVICE)

print(f'Computing SHAP for {n_explain} windows (~{n_explain//10} sec on CPU)...')
if HAS_CAPTUM:
    # GradientShap requires training mode for cuDNN RNN backward pass
    model.train()
    gs = GradientShap(wrapper)
    shap_vals = gs.attribute(inp_tensor, bg_tensor, target=None,
                             n_samples=50, stdevs=0.01)
    shap_np = shap_vals.cpu().detach().numpy()  # [n_explain, LOOKBACK, N_FEAT]
    print(f'SHAP values computed: {shap_np.shape}')
else:
    # Gradient × input fallback
    inp_tensor.requires_grad_(True)
    out = wrapper(inp_tensor)
    out.sum().backward()
    shap_np = (inp_tensor.grad * inp_tensor).cpu().detach().numpy()
    inp_tensor.requires_grad_(False)
    print(f'Gradient×input attribution: {shap_np.shape}')


[data] make_windows: 12099 windows, X=(12099, 21, 4), y=(12099, 14, 2), date range 1981-01-22 → 2014-12-18
SHAP windows: 500 (GPU)
Computing SHAP for 500 windows (~50 sec on CPU)...


SHAP values computed: (500, 21, 4)


## 3 · Feature Importance by Sensor and Lag


In [9]:
# Mean absolute SHAP per feature × lag
mean_abs_shap = np.abs(shap_np).mean(axis=0)   # [LOOKBACK, N_FEAT]
# Convention: array index 0 = t-1 (yesterday / most recent),
#             array index LOOKBACK-1 = t-LOOKBACK (oldest lag).
# S-U1 fix: mean_abs_shap[::-1] so the heatmap x-axis runs
# left=oldest (t-21) → right=most recent (t-1), consistent with Cell 16.
mean_abs_shap_disp = mean_abs_shap[::-1]   # flip: index 0 = oldest lag

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: heatmap of SHAP by lag and feature ──
ax = axes[0]
im = ax.imshow(mean_abs_shap_disp.T, aspect='auto', cmap='magma', interpolation='nearest')
ax.set_yticks(range(N_FEAT))
ax.set_yticklabels(FEATURES, fontsize=10)
# x-axis: leftmost tick = t-LOOKBACK (oldest), rightmost = t-1 (most recent)
xtick_step = max(1, LOOKBACK // 7)
ax.set_xticks(range(0, LOOKBACK, xtick_step))
ax.set_xticklabels([f't-{LOOKBACK - i}' for i in range(0, LOOKBACK, xtick_step)], fontsize=8)
ax.set_xlabel('Lag (days before forecast, 1 = yesterday)', fontsize=10)
ax.set_title('Mean |SHAP| by Feature and Lag', fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.04)

# ── Right: bar chart summed over lags ──
ax2 = axes[1]
feat_importance = mean_abs_shap.sum(axis=0)   # [N_FEAT]
order = np.argsort(feat_importance)[::-1]
colors_feat = sns.color_palette('muted', N_FEAT)
ax2.bar([FEATURES[i] for i in order],
        [feat_importance[i] for i in order],
        color=[colors_feat[i] for i in order])
ax2.set_ylabel('Sum |SHAP| over all lags', fontsize=10)
ax2.set_title('Total Feature Importance (1-day-ahead DO)', fontsize=11)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_shap_feature_importance.png', bbox_inches='tight')
plt.show()


## 4 · Temporal Decay of SHAP

How far back does the LSTM 'look' when predicting DO?


In [10]:
# Sum |SHAP| over all features at each lag
# Convention: lag index 0 = t-1 (yesterday), lag index LOOKBACK-1 = t-LOOKBACK (3 weeks ago).
# This cell reverses the array so the bar chart x-axis matches Cell 14 heatmap:
# leftmost bar = oldest lag (t-LOOKBACK), rightmost bar = most recent (t-1).
shap_by_lag = mean_abs_shap.sum(axis=1)   # [LOOKBACK], index 0 = most recent
lags = np.arange(LOOKBACK, 0, -1)          # most recent lag = LOOKBACK, oldest = 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(LOOKBACK), shap_by_lag[::-1],
       color='#8B4FA8', alpha=0.8, edgecolor='white')
ax.set_xticks(range(0, LOOKBACK, 3))
ax.set_xticklabels([f't-{i+1}' for i in range(0, LOOKBACK, 3)], fontsize=9)
ax.set_xlabel('Lag (days before forecast)', fontsize=11)
ax.set_ylabel('Sum |SHAP| over all features', fontsize=11)
ax.set_title('Temporal Decay of LSTM Input Attributions\n(lag 1 = yesterday, lag 21 = 3 weeks ago)', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_shap_temporal_decay.png', bbox_inches='tight')
plt.show()


## 5 · SHAP Summary Plot (Beeswarm)

Sign of SHAP values tells us the direction of the effect:
positive → higher feature value → higher predicted DO.


In [11]:
# Flatten: [n_explain, LOOKBACK*N_FEAT] — top 10 features
feat_lag_names = [f'{FEATURES[j]}[t-{LOOKBACK-i}]'
                  for i in range(LOOKBACK) for j in range(N_FEAT)]

shap_flat  = shap_np.reshape(n_explain, -1)  # [n_explain, LOOKBACK*N_FEAT]
X_flat     = Xs_test[:n_explain].reshape(n_explain, -1)

# Select top 12 features by mean |SHAP|
# Sort by mean |SHAP| descending — most important first
mean_shap_per_feat = np.abs(shap_flat).mean(axis=0)
top12_idx   = np.argsort(mean_shap_per_feat)[-12:][::-1]  # indices in descending importance order
top12_names = [feat_lag_names[i] for i in top12_idx]

fig, ax = plt.subplots(figsize=(10, 6))
for plot_i, feat_i in enumerate(top12_idx[::-1]):  # enumerate from lowest to highest importance (highest at top of chart)
    sv  = shap_flat[:, feat_i]
    fv  = X_flat[:, feat_i]
    # Normalise feature values to [0,1] for colouring
    fv_norm = (fv - fv.min()) / (fv.ptp() + 1e-9)
    jitter  = np.random.default_rng(SEED + plot_i).uniform(-0.2, 0.2, size=len(sv))
    sc = ax.scatter(sv, np.full_like(sv, plot_i) + jitter,
                    c=fv_norm, cmap='coolwarm', alpha=0.4, s=6, rasterized=True)

ax.set_yticks(range(len(top12_idx)))
# Labels match plot order: position 0 = least important, last position = most important
ax.set_yticklabels([feat_lag_names[i] for i in top12_idx[::-1]], fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value (impact on 1-day-ahead DO)', fontsize=10)
ax.set_title('SHAP Beeswarm — Top 12 Features', fontsize=12)
plt.colorbar(sc, ax=ax, label='Feature value (normalised)')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_shap_beeswarm.png', bbox_inches='tight')
plt.show()


## 6 · Catchment-Attribute SHAP (GBM Surrogate)

Train a LightGBM (or XGBoost fallback) model to predict per-gauge
DO RMSE from catchment attributes, then apply TreeSHAP to explain
which attributes drive predictability across sites.

This requires the multi-site results from notebook 04.


In [12]:
# Try to load multi-site results
ms_path = RESULTS_DIR / 'multisite_results.csv'
if not ms_path.exists():
    print('Multi-site results not found — run notebook 04 first.')
    print('Skipping catchment-attribute SHAP.')
    HAS_MULTISITE = False
else:
    ms_df = pd.read_csv(ms_path)
    transfer_df = ms_df[ms_df['strategy'] == 'transfer_normed'].copy()
    print(f'Loaded multi-site results: {len(transfer_df)} gauges')
    HAS_MULTISITE = True


Loaded multi-site results: 12 gauges


In [13]:
if HAS_MULTISITE:
    meta = load_metadata()
    id_col = next((c for c in meta.columns if 'gauge' in c.lower() and 'id' in c.lower()), meta.columns[0])
    meta['gauge_id_str'] = meta[id_col].astype(str)
    transfer_df['gauge_id'] = transfer_df['gauge_id'].astype(str)
    analysis_df = meta.merge(transfer_df, left_on='gauge_id_str', right_on='gauge_id', how='inner')

    exclude = {'gauge_id', 'gauge_id_str', 'strategy', 'n_windows',
               'rmse_do', 'rmse_temp', 'nse_do', 'kge_do', id_col}
    attr_cols = [
        c for c in analysis_df.columns
        if c not in exclude
        and pd.api.types.is_numeric_dtype(analysis_df[c])
        and analysis_df[c].notna().sum() >= max(5, len(analysis_df) * 0.5)
    ]

    sub = analysis_df[attr_cols + ['rmse_do']].dropna()
    print(f'Samples for GBM: {len(sub)}  |  Attributes: {len(attr_cols)}')
    print('Attributes:', attr_cols[:10])


Samples for GBM: 11  |  Attributes: 19
Attributes: ['gauge_id_x', 'sensor_id', 'nawaf_id', 'nawat_id', 'gauge_easting', 'gauge_northing', 'gauge_lon', 'gauge_lat', 'area', 'area_swiss_perc']


In [14]:
# Ensure training mode for cuDNN RNN gradient computation
model.train()

if HAS_MULTISITE and len(sub) >= 5:
    # Try LightGBM → XGBoost → sklearn GBM fallback
    shap_lib = None
    try:
        import lightgbm as lgb
        import shap as shap_module
        gbm = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                max_depth=3, random_state=SEED, verbose=-1)
        gbm.fit(sub[attr_cols], sub['rmse_do'])
        explainer = shap_module.TreeExplainer(gbm)
        shap_attr  = explainer.shap_values(sub[attr_cols])   # [n, n_attr]
        shap_lib   = 'lightgbm+shap'
        print('Using LightGBM + TreeSHAP')
    except ImportError:
        try:
            import xgboost as xgb
            import shap as shap_module
            gbm = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                                   max_depth=3, random_state=SEED, verbosity=0)
            gbm.fit(sub[attr_cols], sub['rmse_do'])
            explainer = shap_module.TreeExplainer(gbm)
            shap_attr  = explainer.shap_values(sub[attr_cols])
            shap_lib   = 'xgboost+shap'
            print('Using XGBoost + TreeSHAP')
        except ImportError:
            from sklearn.ensemble import GradientBoostingRegressor
            gbm = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                            max_depth=3, random_state=SEED)
            gbm.fit(sub[attr_cols], sub['rmse_do'])
            # Permutation importance as SHAP proxy
            from sklearn.inspection import permutation_importance
            perm = permutation_importance(gbm, sub[attr_cols], sub['rmse_do'],
                                         n_repeats=30, random_state=SEED)
            shap_attr = None
            shap_lib  = 'sklearn_perm'
            print('Using sklearn GBM + permutation importance (install shap for TreeSHAP)')

    if shap_attr is not None and shap_lib is not None:
        mean_abs_attr = np.abs(shap_attr).mean(axis=0)
        attr_order    = np.argsort(mean_abs_attr)[::-1][:15]

        fig, ax = plt.subplots(figsize=(9, 5))
        bar_c = ['#e07b39' if shap_attr[:,i].mean() > 0 else '#4e9ab5'
                 for i in attr_order]
        ax.barh([attr_cols[i] for i in attr_order[::-1]],
                mean_abs_attr[attr_order[::-1]], color=bar_c[::-1])
        ax.set_xlabel('Mean |SHAP|  (impact on DO RMSE)', fontsize=11)
        ax.set_title('Catchment Attribute Importance for DO Predictability\n'
                     f'(method: {shap_lib})', fontsize=11)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / '05_catchment_shap.png', bbox_inches='tight')
        plt.show()
    elif shap_lib == 'sklearn_perm':
        # Permutation importance bar chart
        imp_df = pd.DataFrame({'attribute': attr_cols,
                               'importance': perm.importances_mean})
        imp_df = imp_df.sort_values('importance', ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(imp_df['attribute'][::-1], imp_df['importance'][::-1], color='#8B4FA8')
        ax.set_xlabel('Permutation importance', fontsize=11)
        ax.set_title('Catchment Attribute Permutation Importance', fontsize=11)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / '05_catchment_shap.png', bbox_inches='tight')
        plt.show()
else:
    print('Not enough samples for GBM — need at least 5 gauges with full attribute data.')


Using LightGBM + TreeSHAP


## 7 · SHAP Force Plot for Single Window

Show which inputs pushed the forecast up or down for a specific
representative test window.


In [15]:
# Pick the test window with the largest DO prediction error
from src.model import predict as predict_fn
y_true_test = get_y_true(ds_test, tgt_scaler)
y_pred_test = predict_fn(model, ds_test, tgt_scaler, device=DEVICE)

errors = np.abs(y_true_test[:, 0, 0] - y_pred_test[:, 0, 0])
# O-U3 fix: restrict argmax to [:n_explain] — SHAP values only cover n_explain
# windows, so worst_idx must be within that range or we'd plot the wrong window.
worst_idx = int(np.argmax(errors[:n_explain]))
best_idx  = int(np.argmin(errors[:n_explain]))

print(f'Worst 1-day DO error: {errors[worst_idx]:.3f} mg/L  '
      f'(window start {d_test[worst_idx].date()})')
print(f'Best  1-day DO error: {errors[best_idx]:.3f} mg/L  '
      f'(window start {d_test[best_idx].date()})')

# Waterfall plot of SHAP values for the top 10 features in the worst window
sv_worst = shap_flat[worst_idx] if worst_idx < n_explain else shap_flat[-1]
top10    = np.argsort(np.abs(sv_worst))[-10:][::-1]

fig, ax = plt.subplots(figsize=(9, 4))
c = ['#e07b39' if v > 0 else '#4e9ab5' for v in sv_worst[top10]]
ax.barh([feat_lag_names[i] for i in top10[::-1]],
        sv_worst[top10[::-1]], color=c[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value', fontsize=10)
ax.set_title(f'Top-10 SHAP Attributions — Worst DO Window\n'
             f'({d_test[worst_idx].date() if worst_idx < len(d_test) else "?"})',
             fontsize=11)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_shap_force_worst.png', bbox_inches='tight')
plt.show()


[model] predict: 1348 samples, DO range [-2.34, 2.44] mg/L (scaled)
Worst 1-day DO error: 0.914 mg/L  (window start 2017-05-11)
Best  1-day DO error: 0.000 mg/L  (window start 2017-08-26)


## 8 · Summary


In [16]:
# Ensure training mode for cuDNN RNN gradient computation
model.train()

print('=' * 68)
print('SHAP INTERPRETATION SUMMARY — Gauge 2473')
print('=' * 68)
print(f'Input attribution method : {"GradientSHAP (captum)" if HAS_CAPTUM else "Gradient×Input"}')
print(f'Windows explained        : {n_explain}')
print()
print('Top input features (1-day-ahead DO):')
for i, idx in enumerate(np.argsort(np.abs(shap_flat).mean(axis=0))[-5:][::-1]):
    print(f'  {i+1}. {feat_lag_names[idx]:30s}  mean|SHAP|={np.abs(shap_flat[:,idx]).mean():.4f}')
print('='*68)
print('Key questions for the report:')
print('  - Does the LSTM mostly use recent DO values (short memory)?')
print('  - Does temperature co-vary with DO in expected direction?')
print('  - Which catchment attributes explain cross-site RMSE differences?')


SHAP INTERPRETATION SUMMARY — Gauge 2473
Input attribution method : GradientSHAP (captum)
Windows explained        : 500

Top input features (1-day-ahead DO):
  1. O2C_sensor[t-1]                 mean|SHAP|=0.5798
  2. temp_sensor[t-1]                mean|SHAP|=0.5430
  3. temp_sensor[t-2]                mean|SHAP|=0.2502
  4. O2C_sensor[t-3]                 mean|SHAP|=0.1259
  5. O2C_sensor[t-2]                 mean|SHAP|=0.0962
Key questions for the report:
  - Does the LSTM mostly use recent DO values (short memory)?
  - Does temperature co-vary with DO in expected direction?
  - Which catchment attributes explain cross-site RMSE differences?
